Environment Setup

In [1]:
#Mount google drive
from google.colab import drive
drive.mount('/content/gdrive')
FolderName = '/content/gdrive/My Drive/Colab Notebooks/'

Mounted at /content/gdrive


In [2]:
import pandas as pd
import json

# Load the 4 raw CSV files
orders = pd.read_csv('/content/gdrive/My Drive/Colab Notebooks/orders.csv')
deliveries = pd.read_csv('/content/gdrive/My Drive/Colab Notebooks/deliveries.csv')
app_events = pd.read_csv('/content/gdrive/My Drive/Colab Notebooks/app_events.csv')
complaints = pd.read_csv('/content/gdrive/My Drive/Colab Notebooks/complaints.csv')

In [3]:
#Audit: Find missing values in EVERY dataset
datasets = {"Orders": orders, "Deliveries": deliveries, "App Events": app_events, "Complaints": complaints}

print("--- COMPREHENSIVE DATA QUALITY AUDIT ---")
for name, df in datasets.items():
    missing_counts = df.isnull().sum()
    if missing_counts.sum() > 0:
        print(f"\nMissing values in {name}:")
        print(missing_counts[missing_counts > 0])
    else:
        print(f"\n{name}: No missing values.")

--- COMPREHENSIVE DATA QUALITY AUDIT ---

Missing values in Orders:
booking_channel    25
dtype: int64

Missing values in Deliveries:
delivery_completed_at            19
customer_rating_post_delivery    14
dtype: int64

Missing values in App Events:
order_id    144
dtype: int64

Missing values in Complaints:
compensation_amount    16
dtype: int64


Handling Missing Data

In [4]:
#CLEANING APP EVENTS: Fix the 144 orphaned events
app_events['order_id'] = app_events['order_id'].fillna('SYSTEM_ORPHAN')

#CLEANING COMPLAINTS: Fill missing compensation with 0
if 'compensation_amount' in complaints.columns:
    complaints['compensation_amount'] = complaints['compensation_amount'].fillna(0)

#CLEANING DELIVERIES: Fill missing ratings with the average or 0
if 'customer_rating_post_delivery' in deliveries.columns:
    deliveries['customer_rating_post_delivery'] = deliveries['customer_rating_post_delivery'].fillna(0)

#GLOBAL STANDARDIZATION
#This forces all IDs and Zones to match across all files
for df in [orders, deliveries, app_events, complaints]:
    if 'order_id' in df.columns:
        df['order_id'] = df['order_id'].astype(str).str.strip().str.upper()

#Specific fix for Zone naming consistency
orders['dropoff_zone'] = orders['dropoff_zone'].str.strip().str.upper()
orders['pickup_zone'] = orders['pickup_zone'].str.strip().str.upper()

print("Full Pre-processing Complete. All 4 datasets are now standardized and clean.")

Full Pre-processing Complete. All 4 datasets are now standardized and clean.


Analytical Insight

In [5]:
#Calculate Average Latency per Order
avg_latency = app_events.groupby('order_id')['api_latency_ms'].mean().reset_index()

#Identify orders that have complaints
orders_with_complaints = complaints['order_id'].unique()

#Label orders: 'Had Complaint' vs 'No Complaint'
avg_latency['status'] = avg_latency['order_id'].apply(
    lambda x: 'Complaint Filed' if x in orders_with_complaints else 'No Complaint'
)

#Compare the Average Latency
insight = avg_latency.groupby('status')['api_latency_ms'].mean()

print("--- ANALYTICAL INSIGHT: TECH PERFORMANCE VS SATISFACTION ---")
print(insight)

#Calculate the % difference
diff = ((insight['Complaint Filed'] - insight['No Complaint']) / insight['No Complaint']) * 100
print(f"\nOrders with complaints had {diff:.2f}% higher app latency than successful orders.")

--- ANALYTICAL INSIGHT: TECH PERFORMANCE VS SATISFACTION ---
status
Complaint Filed    484.299828
No Complaint       467.462539
Name: api_latency_ms, dtype: float64

Orders with complaints had 3.60% higher app latency than successful orders.


Creating the JSON file

In [6]:
#Group app events and complaints by ID
events_grouped = app_events.groupby('order_id').apply(lambda x: x.to_dict('records')).to_dict()
complaints_grouped = complaints.groupby('order_id').apply(lambda x: x.to_dict('records')).to_dict()

master_list = []
for _, order_row in orders.iterrows():
    oid = order_row['order_id']
    doc = order_row.to_dict()

    #Nest delivery info (Cleaning check: ensure we handle orders without deliveries)
    delivery_info = deliveries[deliveries['order_id'] == oid].to_dict('records')
    doc['delivery_details'] = delivery_info[0] if delivery_info else {}

    #Nest the arrays
    doc['app_history'] = events_grouped.get(oid, [])
    doc['complaints_history'] = complaints_grouped.get(oid, [])

    master_list.append(doc)

#Save the final file
with open('northstar_final_master.json', 'w') as f:
    json.dump(master_list, f, indent=4)

print("SUCCESS: 'northstar_final_master.json' created. All datasets integrated.")

/tmp/ipykernel_6048/2152971951.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  events_grouped = app_events.groupby('order_id').apply(lambda x: x.to_dict('records')).to_dict()
/tmp/ipykernel_6048/2152971951.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  complaints_grouped = complaints.groupby('order_id').apply(lambda x: x.to_dict('records')).to_dict()


SUCCESS: 'northstar_final_master.json' created. All datasets integrated.
